## **Étape 1 : Base multilingue (FR, EN, ES, ZH)**


1️⃣ Objectif

Permettre au modèle de comprendre et répondre dans 4 langues.

Tester la génération de réponses avant de faire le fine-tuning sur nutrition et style empathique.

2️⃣ Modèle recommandé

Mistral 7B Small → compatible GPU Colab gratuit (~12GB T4/P100).

Alternatives : Falcon Small, Qwen 3B, mT5/XLM-R (si besoin).

3️⃣ Dataset pour test

Pas besoin encore de nutrition, tu peux utiliser un mini dataset Q/A général pour tester les 4 langues :

In [ ]:
# 1️⃣ Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**fine-tuning nutrition**

In [ ]:
!pip install -q transformers datasets accelerate peft

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# ⚡ Paramètres
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
jsonl_path = "/content/nutrition_dataset_complete.jsonl"  # ton JSONL Q/A

In [ ]:
# 1️⃣ Charger le dataset JSONL
dataset = load_dataset("json", data_files=jsonl_path, split="train")

# 2️⃣ Charger le tokenizer et le modèle
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"  # sur GPU Colab
)

In [ ]:

# 3️⃣ Préparer le dataset pour le fine-tuning
def tokenize_fn(examples):
    # On concatène prompt + response pour le fine-tuning causal LM
    return tokenizer(
        [f"{p}\n{r}" for p, r in zip(examples["prompt"], examples["response"])],
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

# 4️⃣ Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# 5️⃣ Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./mistral_nutrition_ft",
    per_device_train_batch_size=1,  # Colab GPU gratuit → batch petit
    gradient_accumulation_steps=4,  # simule un batch plus grand
    num_train_epochs=2,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2
)

# 6️⃣ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 7️⃣ Fine-tuning
trainer.train()

# 8️⃣ Sauvegarde du modèle fine-tuné
model.save_pretrained("./mistral_nutrition_ft")
tokenizer.save_pretrained("./mistral_nutrition_ft")

print("✅ Fine-tuning nutrition terminé !")
